# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. The FAIR² dataset contains mass spectrometry analysis results of extracellular vesicles (EVs) isolated from human mast cells, including rich information about protein abundance, modification, and discovery.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# The dataset.metadata is an object, display basic info with getattr
print(f"Dataset: {getattr(dataset.metadata, 'name', None)}")
print(f"Description\n{getattr(dataset.metadata, 'description', None)}")
print(f"License: {getattr(dataset.metadata, 'license', None)}")
print(f"Citation suggestion: {getattr(dataset.metadata, 'citeAs', None)}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields, referencing entities by their `@id`
print("Available record sets and fields:")
metadata = dataset.metadata

# Collect record_set object(s)
record_sets = getattr(metadata, 'recordSet', [])

record_set_ids = []

for rs in record_sets:
    rs_id = getattr(rs, '@id', None)
    record_set_ids.append(rs_id)
    print(f"- RecordSet @id: {rs_id}")
    # Get fields
    fields = getattr(rs, 'field', [])
    if not isinstance(fields, list):
        fields = [fields]
    for field in fields:
        field_id = getattr(field, '@id', None)
        print(f"    - Field @id: {field_id}")
        # Print columns for each field if present
        columns = getattr(field, 'column', [])
        if not isinstance(columns, list):
            columns = [columns]
        for column in columns:
            col_id = getattr(column, '@id', None)
            print(f"        - Column @id: {col_id}")

# If record_sets is empty, try an alternate way for demo purposes
if not record_set_ids:
    # The recordSet data may sometimes be provided via top-level 'recordSet'.
    # This can be set in the dataset schema as e.g. 'cr:RecordSet'.
    # If not present, we display an informative message and list top-level keys from metadata.
    print("No explicit record sets found in metadata.")
    print("Top-level metadata keys:", dir(metadata))

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Because this Croissant schema may define only a single main record set with all data, we will attempt to extract records from each detected record set. If no explicit record set is present (record_sets is empty), we will attempt to infer the main record set `@id` as described in the FAIR² dataset documentation. Sometimes, the dataset's main RecordSet can be referenced as `'#:main'` or similar. Otherwise, we'll print an error message.

In [ ]:
# Attempt to extract first available record set, fallback to main/default if needed

if record_set_ids:
    # Use all detected record_set @ids
    rs_ids = record_set_ids
else:
    # Try with a default (common) main record set id
    rs_ids = ['#:main'] # this is a fallback, actual id may differ in concrete schemas

dataframes = {}
for rs_id in rs_ids:
    print(f"Attempting to load records from RecordSet @id: '{rs_id}'")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for RecordSet '{rs_id}' with shape: {df.shape}")
        print(f"Columns in '{rs_id}':", df.columns.tolist())
        display(df.head())
    except Exception as e:
        print(f"Could not load records for RecordSet '{rs_id}': {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes. Operations may include removing outliers, transforming data, or summarizing by groups.

**Note:** All references to fields/columns are by their `@id` as described by the Croissant schema.

In [ ]:
# Pick first available DataFrame
if dataframes:
    rs_id = list(dataframes)[0]
    df0 = dataframes[rs_id]
    print(f"Working on DataFrame for RecordSet: {rs_id}")

    # Choose a numeric field: try some common mass spec/protein names
    # We demonstrate here by attempting to pick from standard names
    candidate_ids = [
        'cr:total_peptides',  # example Croissant field id
        'cr:abundance',
        'cr:normalized_abundance',
        'cr:MW',  # molecular weight
        'cr:pI',  # isoelectric point
        'cr:coverage',
    ]
    numeric_field = None
    for fid in candidate_ids:
        if fid in df0.columns:
            numeric_field = fid
            break

    if numeric_field is None:
        print("No standard candidate numeric field found. Columns available:", df0.columns.tolist())
    else:
        print(f"Numeric field chosen for EDA: {numeric_field}")

        # Filter for high-expression (arbitrarily: > mean)
        try:
            df0[numeric_field] = pd.to_numeric(df0[numeric_field], errors='coerce')
            threshold = df0[numeric_field].mean()
            filtered_df = df0[df0[numeric_field] > threshold]
            print(f"Filtered {len(filtered_df)} rows where {numeric_field} > {threshold:.2f}")

            # Normalize numeric field
            filtered_df.loc[:, f"{numeric_field}_normalized"] = (
                filtered_df[numeric_field] - filtered_df[numeric_field].mean()
            ) / filtered_df[numeric_field].std()
            print(f"First rows of normalized field:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Try to group by a candidate group field (e.g., protein family, sample group, etc.)
            group_candidates = ['cr:protein_family', 'cr:sample_label', 'cr:accession']
            group_field = None
            for gf in group_candidates:
                if gf in filtered_df.columns:
                    group_field = gf
                    break
            if group_field:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(f"mean_{numeric_field}")
                print(f"Grouped mean of {numeric_field} by {group_field}:")
                display(grouped_df.head())
            else:
                print("No candidate group field for grouping found.")
        except Exception as e:
            print(f"EDA failed: {e}")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset. Here we plot the distribution of the selected numeric field for the first detected record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'numeric_field' in locals() and numeric_field and dataframes and rs_id in dataframes:
    df0 = dataframes[rs_id].copy()
    # Coerce to numeric for plotting
    df0[numeric_field] = pd.to_numeric(df0[numeric_field], errors='coerce')
    plt.figure(figsize=(8,5))
    sns.histplot(df0[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field} in RecordSet {rs_id}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
else:
    print("No valid numeric field or DataFrame available for visualization.")

## 6. Conclusion
This notebook demonstrated basic exploration of the FAIR² dataset using the `mlcroissant` library. We loaded dataset metadata, listed available record sets and fields, extracted records into pandas DataFrames, performed basic filtering and normalization on protein quantification data, and visualized the distribution of a selected numeric property.

For more advanced analysis, explore the Croissant schema for rich entity relationships, or consult the FAIR² publication for biological context and recommendations.